# Ü8.1 — Inkrementell mit Job Bookmark

Notebook-Gegenstück zum Job-Skript. Der
Bookmark-Schlüssel ist `transformation_ctx` auf Quelle, Mapping und Senke.

**Wichtig — Bookmarks sind ein Job-Runtime-Feature:** der Bookmark-Stand wird pro
**Job-Name** geführt und braucht `--job-bookmark-option job-bookmark-enable`. Im
Notebook setzen wir das über `%%configure` plus `Job.init(<name>)`/`job.commit()`;
primär gehört die Übung in den **Job** (das `.py`). Interaktiv dient das Notebook
dem Nachvollziehen der Code-Form.

**Voraussetzung:** `raw.orders` katalogisiert (Ü3.1).

**So arbeitest du:** Die Code-Zellen sind vorgefertigt und lauffähig — du ersetzt
nur die `____`-Platzhalter. Die Syntax steht schon; du entscheidest nur, *WAS* wohin
gehört. Über jeder Zelle steht je `____` eine Leitfrage.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
# Fülle das ____ aus:
#   ____bookmark_option : Welcher Wert schaltet das Job Bookmark EIN? (aktivieren — nicht pausieren, nicht deaktivieren)
%%configure
{ "--enable-glue-datacatalog": "true", "--job-bookmark-option": "____bookmark_option" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.transforms import ApplyMapping
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)

# Bookmark-Stand wird pro Job-Name gefuehrt — der Name ist der Schluessel.
# Fülle das ____ aus:
#   ____job_name : Wie soll dieser Job heissen? (unter diesem Namen merkt sich das Bookmark den Fortschritt)
job = Job(glueContext)
job.init("____job_name")

OUTPUT = "s3://gfu-glue-training-629452195361/processed/orders/"

## 1) Quelle — `transformation_ctx` trägt den Bookmark

Ohne `transformation_ctx` liest die Quelle bei **jedem** Lauf **alles** neu — das
Bookmark greift nur, wenn jede Stelle ihren eigenen Kontext-Namen hat. Trag unten
`____ctx_quelle` ein.

In [ ]:
# Fülle das ____ aus. Syntax steht schon — du entscheidest nur, WAS wohin gehört.
#   ____ctx_quelle : Wie nennst du den Bookmark-Kontext der QUELLE? (ohne ihn merkt sich das Bookmark den gelesenen Stand NICHT)
source = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="orders", transformation_ctx="____ctx_quelle"
)

## 2) ApplyMapping — den Bookmark-Kontext ergänzen

Auch das Mapping braucht seinen **eigenen, eindeutigen** Kontext-Namen. Trag
`____ctx_mapping` ein.

In [ ]:
# Fülle das ____ aus. Syntax steht schon.
#   ____ctx_mapping : Wie nennst du den Bookmark-Kontext des MAPPINGS? (eigener Name, verschieden von Quelle und Senke)
mapped = ApplyMapping.apply(
    frame=source,
    mappings=[
        ("cust id", "string", "customer_id", "string"),
        ("order total", "string", "order_total", "double"),
        ("order date", "string", "order_date", "string"),
        ("status", "string", "status", "string"),
    ],
    transformation_ctx="____ctx_mapping",
)

## 3) Senke — den Bookmark-Kontext ergänzen

Und die Senke ebenso — auch sie bekommt ihren eigenen Kontext-Namen. Trag
`____ctx_senke` ein.

In [ ]:
# Fülle das ____ aus. Syntax steht schon.
#   ____ctx_senke : Wie nennst du den Bookmark-Kontext der SENKE? (eigener Name, verschieden von Quelle und Mapping)
sink = glueContext.getSink(
    path=OUTPUT,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["order_date"],
    enableUpdateCatalog=True,
    transformation_ctx="____ctx_senke",
)
sink.setCatalogInfo(catalogDatabase="processed", catalogTableName="orders")
sink.setFormat("glueparquet")
sink.writeFrame(mapped)

## 4) commit — schreibt den Bookmark-Stand fort

In [ ]:
job.commit()  # persistiert den Bookmark-Stand fuer den naechsten Lauf

## Test-Ablauf
1. Notebook einmal ausführen (verarbeitet `orders.csv`).
2. `seed/orders_2.csv` nach `raw/orders/` kopieren.
3. Erneut ausführen — nur `orders_2.csv` wird verarbeitet (Bookmark überspringt
   `orders.csv`).

Bookmark zurücksetzen: Glue-Konsole *Reset job bookmark* bzw.
`aws glue reset-job-bookmark --job-name <dein-job-name>` (derselbe Name wie in
`job.init(...)`).